In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [4]:
# 1. Load the Dataset
# ==========================================
print("Loading combined dataset...")
df = pd.read_csv('/content/drive/MyDrive/combined_spam_dataset.csv')

# Remove rows with empty (NaN) text if any exist
df.dropna(subset=['text'], inplace=True)

# Separate features (X) and labels (y)
X = df['text'].astype(str)
y = df['label']

# Split the dataset: 80% for training and 20% for testing
# stratify=y ensures the same proportion of spam/ham in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training data: {len(X_train)} rows | Testing data: {len(X_test)} rows\n")

Loading combined dataset...
Training data: 9454 rows | Testing data: 2364 rows



In [5]:
# 2. Feature Extraction (CountVectorizer)
# ==========================================
print("Converting text to numbers using CountVectorizer...")

# Initialize CountVectorizer to keep the top 5000 most frequent words
vectorizer = CountVectorizer(max_features=5000)

# Learn the vocabulary from the training data and convert it into a numeric matrix
X_train_vec = vectorizer.fit_transform(X_train)

# Convert the testing data into a numeric matrix using the learned vocabulary
# (Note: We only use transform() here, not fit_transform(), to prevent data leakage)
X_test_vec = vectorizer.transform(X_test)

Converting text to numbers using CountVectorizer...


In [6]:
# 3. Train the ML Model (SVM)
# ==========================================
print("Training SVM Model... Please wait.")

# Initialize the Support Vector Machine (SVM) with a linear kernel
svm_model = SVC(kernel='linear', random_state=42)

# Train the model using the training data matrix and labels
svm_model.fit(X_train_vec, y_train)

Training SVM Model... Please wait.


SVC(kernel='linear', random_state=42)

In [7]:
# 4. Evaluate the Model
# ==========================================
print("Evaluating the model...\n")

# Predict labels for the unseen testing data
svm_predictions = svm_model.predict(X_test_vec)

# Calculate the overall accuracy score
svm_acc = accuracy_score(y_test, svm_predictions)

print("="*50)
print(f"✅ SVM Model Accuracy: {svm_acc * 100:.2f}%")
print("="*50)

# Print the detailed classification report (Precision, Recall, F1-Score)
print("\nClassification Report:")
print(classification_report(y_test, svm_predictions))

Evaluating the model...

✅ SVM Model Accuracy: 97.76%

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1652
           1       0.98      0.94      0.96       712

    accuracy                           0.98      2364
   macro avg       0.98      0.97      0.97      2364
weighted avg       0.98      0.98      0.98      2364



In [8]:
# This ensures that the model starts training from the same point every time,
# keeping the accuracy stable when you "Run All".
import numpy as np
import tensorflow as tf
import random

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [9]:
# Import required Deep Learning libraries
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

print("--- Training Deep Learning Model (GRU) ---")

--- Training Deep Learning Model (GRU) ---


In [10]:
# 1. Feature Extraction: Tokenization & Padding
# ==========================================
max_words = 5000    # Consider only the top 5000 most frequent words
max_len = 100       # Pad/truncate every message to a uniform length of 100 words

print("Tokenizing and padding sequences...")
tokenizer = Tokenizer(num_words=max_words)

# Learn the word vocabulary from the training data
tokenizer.fit_on_texts(X_train)

# Convert text sentences into sequences of integers
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences to ensure all sentences have the exact same length (maxlen=100)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

Tokenizing and padding sequences...


In [11]:
# 2. Build GRU Architecture
# ==========================================
print("Building GRU Model...")
gru_model = Sequential([
    # Embedding Layer: Learns word meanings and spatial relationships
    Embedding(input_dim=max_words, output_dim=64, input_length=max_len),

    # GRU Layer: Understands the sequence and context from start to end
    GRU(64, return_sequences=False),

    # Dropout Layer: Randomly drops units to prevent the model from overfitting
    Dropout(0.3),

    # Dense Layer: Outputs the final binary decision (0 = Ham, 1 = Spam)
    Dense(1, activation='sigmoid')
])

Building GRU Model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
# 3. Compile & Train the Model
# ==========================================
# Compile the model with Adam optimizer and binary crossentropy loss
gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Training GRU Model (This may take a minute)...\n")

# Train the model for 10 epochs to improve and stabilize accuracy
history = gru_model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Training GRU Model (This may take a minute)...

Epoch 1/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 17s 100ms/step - accuracy: 0.7904 - loss: 0.5017 - val_accuracy: 0.8129 - val_loss: 0.4677
Epoch 2/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 13s 100ms/step - accuracy: 0.8588 - loss: 0.3991 - val_accuracy: 0.9006 - val_loss: 0.3086
Epoch 3/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 21s 100ms/step - accuracy: 0.9117 - loss: 0.2960 - val_accuracy: 0.9450 - val_loss: 0.2091
Epoch 4/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 14s 102ms/step - accuracy: 0.9397 - loss: 0.2187 - val_accuracy: 0.9567 - val_loss: 0.1762
Epoch 5/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 19s 87ms/step - accuracy: 0.9522 - loss: 0.1838 - val_accuracy: 0.9567 - val_loss: 0.1759
Epoch 6/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.9187 - loss: 0.2704 - val_accuracy: 0.8298 - val_loss: 0.4499
Epoch 7/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 21s 103ms/step - accuracy: 0.8451 - loss: 0.4163 - val_accuracy: 0.8668 - val_loss: 0.3688
Epoch 8/10
133/133 ━━━━━━━━━━━━━━━━━━

In [14]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# 1. Predictions ලබාගැනීම
y_pred_gru_prob = gru_model.predict(X_test_pad)
y_pred_gru = (y_pred_gru_prob > 0.5).astype(int)

# 2. Print Evaluation Metrics
print("="*50)
print("=== GRU Model Evaluation Results ===")
print("Accuracy :", accuracy_score(y_test, y_pred_gru))
print("Precision:", precision_score(y_test, y_pred_gru))
print("Recall   :", recall_score(y_test, y_pred_gru))
print("F1-Score :", f1_score(y_test, y_pred_gru))
print("="*50)

# 3. Detailed Classification Report (Optional)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_gru, digits=4))

74/74 ━━━━━━━━━━━━━━━━━━━━ 6s 66ms/step
=== GRU Model Evaluation Results ===
Accuracy : 0.8752115059221658
Precision: 0.7165109034267912
Recall   : 0.9691011235955056
F1-Score : 0.8238805970149253

Detailed Classification Report:
              precision    recall  f1-score   support

           0     0.9843    0.8347    0.9034      1652
           1     0.7165    0.9691    0.8239       712

    accuracy                         0.8752      2364
   macro avg     0.8504    0.9019    0.8636      2364
weighted avg     0.9036    0.8752    0.8794      2364

